# Convert Snowflake Arctic Embed XS → float32 TFLite
Converts the HuggingFace model to a CPU-compatible float32 `.tflite` file for WulfPak.
Run cells top-to-bottom. GPU not required. Takes ~5 min on Colab free tier.

In [ ]:
# Cell 1 — Install dependencies (~2 min)
!pip install -q "optimum[onnxruntime]" onnx2tf onnx tensorflow

In [ ]:
# Cell 2 — Export model to ONNX via Optimum (downloads ~90 MB)
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer
import os

MODEL_ID = "Snowflake/snowflake-arctic-embed-xs"

print("Exporting to ONNX...")
model = ORTModelForFeatureExtraction.from_pretrained(MODEL_ID, export=True)
model.save_pretrained("/content/arctic_onnx")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.save_pretrained("/content/arctic_onnx")

print("Done. Files:")
for f in os.listdir("/content/arctic_onnx"):
    size = os.path.getsize(f"/content/arctic_onnx/{f}") / 1024 / 1024
    print(f"  {f}  ({size:.1f} MB)")

In [ ]:
# Cell 3 — Convert ONNX → TFLite float32
import onnx2tf, os

os.makedirs("/content/arctic_tflite", exist_ok=True)

print("Converting...")
onnx2tf.convert(
    input_onnx_file_path="/content/arctic_onnx/model.onnx",
    output_folder_path="/content/arctic_tflite",
    non_verbose=True,
)

print("Output files:")
for f in os.listdir("/content/arctic_tflite"):
    size = os.path.getsize(f"/content/arctic_tflite/{f}") / 1024 / 1024
    print(f"  {f}  ({size:.1f} MB)")

In [ ]:
# Cell 4 — Inspect tensor names (paste this output into the chat so Kotlin code can be updated)
import tensorflow as tf, glob

tflite_files = glob.glob("/content/arctic_tflite/*.tflite")
assert tflite_files, "No .tflite found — check Cell 3 output"
tflite_path = tflite_files[0]
print(f"Using: {tflite_path}\n")

interp = tf.lite.Interpreter(model_path=tflite_path)
interp.allocate_tensors()

print("=== INPUT TENSORS ===")
for t in interp.get_input_details():
    print(f"  [{t['index']}]  name='{t['name']}'  shape={t['shape']}  dtype={t['dtype']}")

print("\n=== OUTPUT TENSORS ===")
for t in interp.get_output_details():
    print(f"  [{t['index']}]  name='{t['name']}'  shape={t['shape']}  dtype={t['dtype']}")

In [ ]:
# Cell 5 — Sanity-check: semantic similarity test
import numpy as np

SEQ_LEN = 128

def embed(text: str) -> np.ndarray:
    enc = tokenizer(text, return_tensors="np", max_length=SEQ_LEN,
                    padding="max_length", truncation=True)
    for t in interp.get_input_details():
        name = t["name"]
        if   "input_ids"       in name: interp.set_tensor(t["index"], enc["input_ids"].astype(t["dtype"]))
        elif "attention_mask"  in name: interp.set_tensor(t["index"], enc["attention_mask"].astype(t["dtype"]))
        elif "token_type_ids" in name: interp.set_tensor(t["index"], enc["token_type_ids"].astype(t["dtype"]))
    interp.invoke()
    last_hidden = interp.get_tensor(interp.get_output_details()[0]["index"])  # [1, 128, 384]
    mask = enc["attention_mask"][0].astype(np.float32)
    return (last_hidden[0] * mask[:, None]).sum(0) / mask.sum()  # mean pool → [384]

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

e1 = embed("had dinner at Sharkey's with friends last night")
e2 = embed("went out for food and drinks with the group")
e3 = embed("quarterly earnings report and budget forecast")

print(f"dinner <-> dinner : {cosine(e1, e1):.4f}  (expect 1.0000)")
print(f"dinner <-> food   : {cosine(e1, e2):.4f}  (expect high >0.7)")
print(f"dinner <-> finance: {cosine(e1, e3):.4f}  (expect low  <0.4)")

assert cosine(e1, e2) > cosine(e1, e3), "Similarity ordering wrong — model may not have converted correctly"
print("\nSanity check passed!")

In [ ]:
# Cell 6 — Download
# Drop the .tflite into core-logic/src/main/assets/ replacing the float16 file.
# vocab.txt does NOT need replacing — same BERT WordPiece tokenizer.
import shutil, os

output_path = "/content/snowflake-arctic-embed-xs.tflite"
shutil.copy(tflite_path, output_path)
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    # Kaggle: file appears in the Output tab
    shutil.copy(output_path, "/kaggle/working/snowflake-arctic-embed-xs.tflite")
    print("Kaggle: grab the file from the Output tab on the right.")